In [ ]:
!pip install datasets transformers


Access the travel-related dataset directly from Hugging Face

In [ ]:
from google.colab import drive
import os
import torch

# Mount Google Drive
drive.mount('/content/drive')

In [5]:
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

# Step 1: Load the Kaggle Dataset
dataset_path = "dialogs.txt"

# Load as a tab-separated file
dialogs_df = pd.read_csv(dataset_path, sep="\t", header=None, names=["input", "output"])

# Convert the DataFrame to Hugging Face Dataset
dialogs_hf_dataset = Dataset.from_pandas(dialogs_df)

# Verify the Kaggle dataset
print("Sample from dialogs dataset:", dialogs_hf_dataset[0])

# Step 2: Load Hugging Face Datasets
dataset1 = load_dataset("JasleenSingh91/travel-questions-response")
dataset2 = load_dataset("bitext/Bitext-travel-llm-chatbot-training-dataset")
dataset3 = load_dataset("nitv/fitness_chatbot_245")

# Combine Hugging Face datasets
huggingface_combined = concatenate_datasets([
    dataset1["train"],
    dataset2["train"],
    dataset3["train"],
])

# Verify Hugging Face combined dataset
print("Sample from Hugging Face combined dataset:", huggingface_combined[0])\

print("Sample from dialogs_hf_dataset dataset:", dialogs_hf_dataset[0])

# Step 3: Combine with Kaggle Dataset
combined_dataset = concatenate_datasets([huggingface_combined, dialogs_hf_dataset])

# Verify the final combined dataset
print(f"Combined dataset size: {len(combined_dataset)}")
print("Sample from combined dataset:", combined_dataset[0])

# Step 4: Save the Combined Dataset
combined_dataset.save_to_disk("final_combined_dataset")

# Step 5: Download the Saved Dataset (Optional for Colab Users)
import shutil
shutil.make_archive("final_combined_dataset", 'zip', "final_combined_dataset")

# If you're using Colab, you can download it using:
from google.colab import files
#files.download("final_combined_dataset.zip")


Sample from dialogs dataset: {'input': 'hi, how are you doing?', 'output': "i'm fine. how about yourself?"}
Sample from Hugging Face combined dataset: {'Input': 'Can you suggest the best travel destinations for my next trip?', 'Response_1': ' Absolutely! To help me suggest the best travel destinations for your next trip, I\'ll need to know a few things from you:\n1. What type of trip are you looking for? (e.g., beach vacation, adventure travel, city break, cultural tour, etc.)\n2. What is your preferred climate? (e.g., warm, cold, tropical, dry, etc.)\n3. How long will you be traveling for?\n4. What is your budget range?\n5. Do you have any specific travel preferences, such as accessibility for individuals with disabilities or dietary restrictions?\nBased on your answers to these questions, I can suggest some travel destinations that may fit your needs and preferences. Here are a few options to consider:\n1. Maui, Hawaii: If you\'re looking for a warm, tropical beach vacation, Maui is 

Saving the dataset (0/1 shards):   0%|          | 0/41637 [00:00<?, ? examples/s]

In [6]:
print(f"Combined dataset size: {len(combined_dataset)}")
for i in range(0,len(combined_dataset), 300):
  print("Sample from combined dataset:", combined_dataset[i] ," -->", i)

Combined dataset size: 41637
Sample from combined dataset: {'Input': 'Can you suggest the best travel destinations for my next trip?', 'Response_1': ' Absolutely! To help me suggest the best travel destinations for your next trip, I\'ll need to know a few things from you:\n1. What type of trip are you looking for? (e.g., beach vacation, adventure travel, city break, cultural tour, etc.)\n2. What is your preferred climate? (e.g., warm, cold, tropical, dry, etc.)\n3. How long will you be traveling for?\n4. What is your budget range?\n5. Do you have any specific travel preferences, such as accessibility for individuals with disabilities or dietary restrictions?\nBased on your answers to these questions, I can suggest some travel destinations that may fit your needs and preferences. Here are a few options to consider:\n1. Maui, Hawaii: If you\'re looking for a warm, tropical beach vacation, Maui is a great option. With beautiful beaches, fantastic weather, and a variety of activities, Maui

preprocessing

In [109]:
import random
import pandas as pd
from datasets import Dataset

def standardize_format(example):
    # Adjust based on column names in the datasets
    if "Input" in example and example['Input']!=None and "Response_1" in example and example['Response_1']!=None:
        return {"input": example["Input"], "output": example["Response_1"]}
    elif "questions" in example and example['questions']!=None and "answers" in example and example['answers']!=None:
        return {"input": example["questions"], "output": example["answers"]}
    elif "question" in example and example['question']!=None and "answer" in example and example['answer']!=None:
        return {"input": example["question"], "output": example["answer"]}
    else:
        return {"input": example["input"], "output": example["output"]}

# Apply standardization
standardized_dataset = combined_dataset.map(standardize_format, remove_columns=combined_dataset.column_names)

# Remove any None values
standardized_dataset = standardized_dataset.filter(lambda x: x is not None)

def filter_relevant_examples(example):
    # Check if the 'input' field exists and is not None
    if example.get("input") and isinstance(example["input"], str):
        return True
    return False   #Exclude examples without a valid 'input' field


# Apply the filter to the dataset

filtered_dataset = standardized_dataset.filter(filter_relevant_examples)

# Adapt responses to the running domain (NEED TO CHANGE !!!)
def adapt_responses(example):
    # Define keyword categories and their corresponding running-related responses
    response_map = {
        "travel": "This location offers fantastic running routes. Let me suggest one for you!",
        "trip": "This place has amazing running paths. Let me find a route for your trip!",
        "vacation": "This destination has great options for running. Let me help you find a route!",
        "hiking": "Looking for a trail? Let me find you a running route in this area!",
        "trekking": "You can discover beautiful running trails here. Let me assist you!",
        "activities": "This location is perfect for running activities. Let me plan a route for you!",
        "forest": "There are serene running trails in the forest. Let me pick one for you!",
        "park": "Parks often have great running paths. Let me find the best one for you!",
        "hotel": "This location offers fantastic running routes. Let me suggest one for you!",
        "resort": "This destination has excellent options for running routes. Let me plan something for you!",
        "beach": "Coastal routes are wonderful for running. Let me find a scenic one for you!",
        "adventure": "This location is ideal for running adventures. Let me suggest a route!"
    }

    # Define words to replace in the input text
    replacement_map = {
        "trip": "route",
        "travel": "running adventure",
        "vacation": "running getaway",
        "hiking": "trail running",
        "trekking": "trail running",
        "activities": "running activities",
        "resort": "running destination",
        "hotel": "rest stop for runners",
        "beach": "coastal running path",
        "adventure": "running experience",
        "destinations": "running routes",
    }
    # Check keywords in input and adapt the response
    for keyword, response in response_map.items():
        if keyword in example["input"].lower():
            example["output"] = response
            return example
    # Replace words in the input text
    for word, replacement in replacement_map.items():
        if word in example["input"].lower():
            example["input"] = example["input"].lower().replace(word, replacement)
    return example

adapted_dataset = filtered_dataset.map(adapt_responses)


# Generate synthetic running route data
def generate_synthetic_data(num_examples=500):
    locations = [
        "work", "current location", "home", "park", "gym",
        "school", "mall", "train station", "lake", "forest"
    ]
    terrains = [
        "trail", "flat", "scenic", "urban", "hilly",
        "grassy", "rocky", "coastal", "desert", "wooded"
    ]
    messages = [
        "Wow", "Good luck!", "Sounds great", "Have fun!",
        "Enjoy your run!", "Keep going!", "You can do it!",
        "Stay safe!", "Happy running!", "Amazing choice!"
    ]
    opens = [
        "Find me", "Suggest", "Give me", "Provide me with",
        "Recommend", "Help me find", "Show me", "Locate",
        "Plan", "Pick"
    ]

    synthetic_data = []
    for i in range(1, num_examples + 1):  # Start from 1 for better formatting
        location = random.choice(locations)
        terrain = random.choice(terrains)
        message = random.choice(messages)
        open = random.choice(opens)
        input_text = f"{open} a {i} km {terrain} route near {location}."
        output_text = message
        synthetic_data.append({"input": input_text, "output": output_text})

    return synthetic_data

synthetic_dataset = generate_synthetic_data()
synthetic_df = pd.DataFrame(synthetic_dataset)

# Combine adapted and synthetic datasets
adapted_df = pd.DataFrame(adapted_dataset)
augmented_dataset = pd.concat([adapted_df, synthetic_df]).reset_index(drop=True)

# Convert to Hugging Face Dataset format
final_dataset = Dataset.from_pandas(augmented_dataset)

# Save the final dataset for reuse
final_dataset.save_to_disk("final_running_chatbot_dataset")

# Verify the dataset
print(f"Final dataset size: {len(final_dataset)}")
print(f"Sample entry: {final_dataset[0]}")


Saving the dataset (0/1 shards):   0%|          | 0/10234 [00:00<?, ? examples/s]

Final dataset size: 10234
Sample entry: {'output': 'This location offers fantastic running routes. Let me suggest one for you!', 'input': 'Can you suggest the best travel destinations for my next trip?'}


In [8]:
for i in range(0,len(final_dataset), 200):
  print("Sample from combined dataset:", final_dataset[i] ," -->", i)

Sample from combined dataset: {'output': 'This location offers fantastic running routes. Let me suggest one for you!', 'input': 'Can you suggest the best travel destinations for my next trip?'}  --> 0
Sample from combined dataset: {'output': 'Coastal routes are wonderful for running. Let me find a scenic one for you!', 'input': 'Can you recommend a mid-range hostel near the beach?'}  --> 200
Sample from combined dataset: {'output': 'This location offers fantastic running routes. Let me suggest one for you!', 'input': 'Can you recommend a mid-range hotel in the USA that offers a comfortable stay with great value?'}  --> 400
Sample from combined dataset: {'output': 'Looking for a trail? Let me find you a running route in this area!', 'input': 'What are the best destinations for spring hiking, with beautiful landscapes and mild weather?'}  --> 600
Sample from combined dataset: {'output': 'This location is perfect for running activities. Let me plan a route for you!', 'input': 'Can you rec

In [110]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Add new tokens derived from the dataset
new_tokens = list(set(final_dataset["input"] + final_dataset["output"]))
new_tokens = [f" {token}" for token in new_tokens if not token.startswith(" ")]  # Add leading spaces
tokenizer.add_tokens(new_tokens)

# Tokenize the dataset
def tokenize_function(batch):
    # Tokenize input
    tokenized_input = tokenizer(
        batch["input"],
        max_length=100,
        padding="max_length",  # Pad to max length
        truncation=True        # Truncate long sequences
    )

    # Tokenize output
    tokenized_output = tokenizer(
        batch["output"],
        max_length=100,
        padding="max_length",  # Pad to max length
        truncation=True        # Truncate long sequences
    )

    return {
        "input_ids": tokenized_input["input_ids"],
        "attention_mask": tokenized_input["attention_mask"],
        "labels": tokenized_output["input_ids"],  # Tokenized labels
    }

# Apply tokenization to the dataset
tokenized_dataset = final_dataset.map(tokenize_function, batched=True)

# Verify tokenizer performance
print(f"Vocabulary size after adding tokens: {len(tokenizer)}")
print("Sample tokenized entry:", tokenized_dataset[0])


Map:   0%|          | 0/10234 [00:00<?, ? examples/s]

Vocabulary size after adding tokens: 43199
Sample tokenized entry: {'output': 'This location offers fantastic running routes. Let me suggest one for you!', 'input': 'Can you suggest the best travel destinations for my next trip?', 'input_ids': [1072, 25, 3130, 8, 200, 1111, 10944, 21, 82, 416, 1469, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [100, 1128, 704, 2723, 1180, 9729, 5, 1563, 140, 3130, 80, 21, 25, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Verification
After applying the map function, verify the tokenized dataset:

In [111]:
# Check a tokenized sample
print("Tokenized sample:", tokenized_dataset[0])

# Check the dataset schema to ensure consistent lengths
print("Dataset features:", tokenized_dataset.features)


Tokenized sample: {'output': 'This location offers fantastic running routes. Let me suggest one for you!', 'input': 'Can you suggest the best travel destinations for my next trip?', 'input_ids': [1072, 25, 3130, 8, 200, 1111, 10944, 21, 82, 416, 1469, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [100, 1128, 704, 2723, 1180, 9729, 5, 1563, 140, 3130, 80, 21, 25, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [112]:
# Print a few examples from the dataset
print("Raw Dataset Examples:")
for i in range(5):
    print(f"Example {i + 1}:")
    print(f"Input: {final_dataset[i]['input']}")
    print(f"Output: {final_dataset[i]['output']}")
    print()

Raw Dataset Examples:
Example 1:
Input: Can you suggest the best travel destinations for my next trip?
Output: This location offers fantastic running routes. Let me suggest one for you!

Example 2:
Input: What activities and places should I explore in the USA?
Output: This location is perfect for running activities. Let me plan a route for you!

Example 3:
Input: can you recommend relaxing running routes to visit during the winter?
Output:  Absolutely! Winter can be a beautiful and peaceful time to travel, and there are several destinations around the world known for their relaxing atmospheres during this season. Here are some recommendations:
1. Whistler, Canada: This scenic mountain town in British Columbia is famous for its skiing and snowboarding, but it's also a great place to unwind. Relax in a hot tub overlooking the snowy mountains, or take a leisurely stroll through the charming village.
2. Hakone, Japan: Located near Mount Fuji, Hakone is known for its hot springs, or "onsen.

Split into Training, Validation, and Test Sets: Use an 80-10-10 split:

In [82]:
# Print tokenized examples
print("Tokenized Dataset Examples:")
for i in range(5):
    print(f"Example {i + 1}:")
    print(f"Input IDs: {tokenized_dataset[i]['input_ids']}")
    print(f"Attention Mask: {tokenized_dataset[i]['attention_mask']}")
    print(f"Labels: {tokenized_dataset[i]['labels']}")
    print()

Tokenized Dataset Examples:
Example 1:
Input IDs: [1072, 25, 3130, 8, 200, 1111, 10944, 21, 82, 416, 1469, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Labels: [100, 1128, 704, 2723, 1180, 9729, 5, 1563, 140, 3130, 80, 21, 25, 55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [83]:
# Print actual tokenized text
for i in range(0,len(tokenized_dataset),300):
    input_tokens = tokenizer.convert_ids_to_tokens(tokenized_dataset[i]["input_ids"])
    output_tokens = tokenizer.convert_ids_to_tokens(tokenized_dataset[i]["labels"])
    print(f"Input Tokens: {input_tokens}")
    print(f"Output Tokens: {output_tokens}")
    print()

Input Tokens: ['▁Can', '▁you', '▁suggest', '▁the', '▁best', '▁travel', '▁destinations', '▁for', '▁my', '▁next', '▁trip', '?', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Output Tokens: ['▁This', '▁location', '▁offers', '▁fantastic', '▁running', '▁routes

In [113]:
from transformers import T5ForConditionalGeneration
import torch

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Expand the embedding layer to include new tokens
model.resize_token_embeddings(len(tokenizer))

# Ensure embeddings are trainable
model.shared.requires_grad = True  # Fine-tune embeddings

# Move the model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Print model architecture
print(model)


T5ForConditionalGeneration(
  (shared): Embedding(43199, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(43199, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [114]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # Extract fields
    input_ids = [torch.tensor(item["input_ids"]) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"]) for item in batch]
    labels = [torch.tensor(item["labels"]) for item in batch]

    # Dynamically pad sequences
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=tokenizer.pad_token_id)

    # Return as a dictionary
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [115]:
from torch.utils.data import DataLoader

# Split the dataset into training and validation sets
train_test_split = tokenized_dataset.train_test_split(test_size=0.1)

# Separate the train and validation datasets
train_dataset = train_test_split['train']
valid_dataset = train_test_split['test']

# Create DataLoaders for the train and validation sets
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, collate_fn=collate_fn)

print(f"Training set size: {len(train_loader)}")
print(f"Validation set size: {len(valid_loader)}")




Training set size: 144
Validation set size: 16


Verify the splits

In [116]:
# Check the sizes of datasets
print(f"Train dataset size: {len(train_loader)}")

# Example: Check the first batch from train_loader
for batch in train_loader:
    print("Train batch example:", batch)
    break

Train dataset size: 144
Train batch example: {'input_ids': tensor([[   25,   708,  3182,  ...,     0,     0,     0],
        [  180, 13917,   222,  ...,     0,     0,     0],
        [  363,  1087,    54,  ...,     0,     0,     0],
        ...,
        [ 2087,     3,    23,  ...,     0,     0,     0],
        [  180, 13917,   222,  ...,     0,     0,     0],
        [   25,   278,    31,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[  34,   47,   30,  ...,    0,    0,    0],
        [ 100, 1128,  704,  ...,    0,    0,    0],
        [ 100, 1128,   19,  ...,    0,    0,    0],
        ...,
        [ 150,    6,   25,  ...,    0,    0,    0],
        [ 100, 3954,   65,  ...,    0,    0,    0],
        [  13,  503,    3,  ...,    0,    0,    0]])}


In [ ]:
!pip install evaluate
!pip install sacrebleu


In [ ]:
from transformers import AdamW, DataCollatorForSeq2Seq

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

import evaluate

# Load the SacreBLEU metric
metric = evaluate.load("sacrebleu")

# Define a function to compute metrics
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # Clip predictions and labels to the valid token ID range to avoid OverflowError
    predictions = predictions.clip(0, len(tokenizer) - 1)  # Clip predictions
    labels = labels.clip(0, len(tokenizer) - 1)            # Clip labels

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # SacreBLEU expects newline-separated sentences
    decoded_preds = ["\n".join(pred.split()) for pred in decoded_preds]
    decoded_labels = [["\n".join(label.split())] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}


# Initialize the data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True  # Enable dynamic padding
)



In [118]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Define training arguments for the Seq2SeqTrainer
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-running-routes",  # Directory to save model checkpoints
    evaluation_strategy="epoch",      # Evaluate the model at the end of every epoch
    learning_rate=3e-5,               # Learning rate for the optimizer
    per_device_train_batch_size=64,    # Batch size for training
    per_device_eval_batch_size=64,     # Batch size for evaluation
    weight_decay=0.01,                # Weight decay for regularization
    save_total_limit=3,               # Keep the last 3 checkpoints only
    num_train_epochs=35,              # Total number of training epochs
    predict_with_generate=True,       # Use the model's generate method for predictions
    logging_dir="./logs",             # Directory for logging metrics and progress
    logging_steps=10,                 # Log metrics every 10 steps
    save_strategy="epoch",            # Save the model at the end of each epoch
    load_best_model_at_end=True,      # Automatically load the best model (based on evaluation metrics) at the end
    report_to="none"
)

# Initialize the Hugging Face Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,                      # The pre-trained or fine-tuned model
    args=training_args,               # Training arguments defined above
    train_dataset=train_dataset,      # Training dataset (prepared earlier)
    eval_dataset=valid_dataset,       # Validation dataset (prepared earlier)
    data_collator=data_collator,      # Use the data collator for dynamic padding
    compute_metrics=compute_metrics,  # Function to compute BLEU during evaluation
)


# Train the model
trainer.train()



/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Bleu
1,0.982500,0.799861,0.000000
2,0.673800,0.552254,6.360393
3,0.595300,0.480058,13.985250
4,0.511300,0.453214,16.813348
5,0.442600,0.435912,17.213804
6,0.463400,0.423917,18.101054
7,0.442300,0.414163,19.613024
8,0.446100,0.406333,21.335481
9,0.465700,0.399811,23.386082
10,0.403500,0.394684,23.483692


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=5040, training_loss=0.4995920970799431, metrics={'train_runtime': 1318.5157, 'train_samples_per_second': 244.479, 'train_steps_per_second': 3.822, 'total_flos': 8520982364160000.0, 'train_loss': 0.4995920970799431, 'epoch': 35.0})

In [119]:
def generate_response(input_text, max_length=50):
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_length=max_length)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

user_input = [
    "How are you?",
    "Find me a 10 km trail near my current location.",
    "I am looking for a flat 3 km route near my home.",
    "Can you suggest a scenic route for 5 km?",
    "What is the best running route near the gym?",
    "Where can I find a trail to run 7 km?",
    "I need a route to run 5 km.",
    "Can you find me a hilly 8 km route?",
    "Show me a trail near the lake for a 6 km run.",
    "I want a flat route near the park for 4 km.",
    "What’s the best 10 km running route near downtown?",
    "Suggest a running path near my office for 7 km.",
    "Find me a coastal route to run 3 km.",
    "I’d like to run 12 km in a forest trail.",
    "Can you locate a grassy route for 9 km?",
    "What’s the closest scenic route for a quick 2 km jog?",
    "I need a trail running route for 10 km.",
    "Show me a rocky 8 km running path.",
    "Can you suggest an urban route for 5 km?",
    "Find a wooded area for a 6 km jog.",
    "What’s the best desert trail for running 7 km?",
    "Suggest a 5 km run near the city center.",
    "Find me a running route near the mall.",
    "Locate a running track near the train station for 3 km.",
    "Plan a 4 km running route near the beach.",
    "What’s a good 2 km route for a morning jog near my school?",
    "Give me a 6 km scenic trail near the forest reserve.",
    "Find a flat route for 8 km near my neighborhood.",
    "Can you help me find a grassy route near the gym?",
    "What’s the best running trail near the lake for 10 km?"
]
for inp in user_input:
  response = generate_response(inp)
  print(f"Input: {inp}")
  print(f"Model Response: {response}")


Input: How are you?
Model Response: i'm not sure.
Input: Find me a 10 km trail near my current location.
Model Response: Looking for a trail? Let me find you a running route in this area!
Input: I am looking for a flat 3 km route near my home.
Model Response: I'd be happy to help you find a flat 3 km route near my home.
Input: Can you suggest a scenic route for 5 km?
Model Response: Absolutely! I'd be happy to help you plan a scenic route for 5 km. Here are some suggestions: 1. New England, USA: This beautiful and scenic route is a must-visit for anyone looking for a scenic
Input: What is the best running route near the gym?
Model Response: Absolutely! I'd be happy to help you plan a trip to the gym. Here are some of the best places to go to the gym: 1. The Gym in the Gym: The Gym is a popular gym area
Input: Where can I find a trail to run 7 km?
Model Response: Looking for a trail? Let me find you a running route in this area!
Input: I need a route to run 5 km.
Model Response: i'm gla

In [120]:
# Specify the save path in your Google Drive
save_directory = "/content/drive/My Drive/gug's_best_model_custom_seq2seq_model_with_T5"

# Create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# Save the model's state_dict (weights)
torch.save(model.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

# Save the tokenizer
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer saved to Google Drive at {save_directory}")


Model and tokenizer saved to Google Drive at /content/drive/My Drive/gug's_best_model_custom_seq2seq_model_with_T5


In [69]:
def interactive_evaluation():
    """
    Interactively evaluate the trained model.
    Args:
        model: Trained Seq2Seq model.
        tokenizer: Tokenizer used for encoding/decoding.
        max_length: Maximum length of the generated response.
    """
    print("Interactive Evaluation: Type a question and press Enter (type 'quit' to exit).")

    while True:
        question = input("You: ")
        if question.lower() == "quit":
            print("Exiting evaluation. Goodbye!")
            break

        # Decode the response
        response = generate_response(question)
        print(f"Model: {response}")


In [ ]:
interactive_evaluation(model)


Interactive Evaluation: Type a question and press Enter (type 'quit' to exit).
You: Hi there, can you plan me a running route?
Model: i'll give you a speech.
You: i thinking around 10 km.
Model: i'm not sure.
You: simple sounds good
Model: greats
You: I want to start running from my current location
Model: i ideal for running. Let me
You: i'd like to end at Haneviim 37
Model: i'll bet i'll just wait.
You: quit
Exiting evaluation. Goodbye!
